# Teks — LLM kecil (Qwen) + LoRA sebagai pengklasifikasi, k-fold + OOF
**Kapan dipakai:** klasifikasi teks dengan konteks/nuansa sulit (sarkasme, topik campur, teks
panjang), ada GPU, dan sudah ada `finetune_oof` (BERT) sebagai pembanding. LLM decoder sering
menang beberapa poin di Kaggle NLP terbaru dan hasilnya beda jenis dengan BERT, jadi bagus untuk
ensemble. Bukan prompting: kepala klasifikasi dilatih seperti BERT.

**Cara:** bobot LLM dibekukan (bf16), hanya matriks LoRA kecil (rank `LORA_R`) di tiap Linear
attention/MLP + kepala `score` yang dilatih. LoRA ditulis sendiri (kelas `LoraLinear`), tanpa
paket `peft`. Prediksi diambil dari token terakhir yang bukan padding.

**Alur:** Konfigurasi → baca → split → tokenisasi → model + LoRA → latih tiap fold (simpan LoRA
terbaik) → OOF → simpan → submission → validator.

**Saat lomba yang diubah:** sel Konfigurasi. VRAM 8 GB: model 0.5–0.6B dengan `BATCH_SIZE` 8 dan
`GRAD_CHECKPOINT=True` aman. Waktu mepet: `FOLDS_TO_RUN = [0]`.

**Data demo:** SmSA (`sentiment`, 3 kelas).

In [ ]:
import html
import os
import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold
from torch import nn
from torch.nn.functional import cross_entropy, softmax
from torch.nn.utils import clip_grad_norm_
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import get_linear_schedule_with_warmup

## Konfigurasi

In [ ]:
DATA_FOLDER = "../../../data_latihan/text/sentiment"          # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/smsa_train.csv"                    # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/smsa_test.csv"                      # GANTI saat lomba
OUTPUT_FOLDER = "../../../outputs/text/llm_lora_klasifikasi"
OOF_FOLDER = "../../../outputs/text/oof"
SAMPLE_CSV = f"{OUTPUT_FOLDER}/sample_submission_demo.csv"     # GANTI saat lomba: file panitia
SUBMISSION_NAME = "submission.csv"
ANSWER_CSV = f"{DATA_FOLDER}/smsa_test_labels.csv"             # HAPUS saat lomba (kunci demo)
ID_COL = "id"                    # GANTI saat lomba
TEXT_COL = "text"                # GANTI saat lomba
TARGET = "label"                 # GANTI saat lomba
SUBMIT_COL = TARGET
N_FOLDS = 5
SPLIT_SEED = 42
NOTEBOOK = "llm_lora_klasifikasi"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
PRETRAINED = "../../../pretrained/qwen_05b"           # GANTI: Qwen2.5-0.5B-Instruct
# PRETRAINED = "../../../pretrained/qwen3_06b"   # Qwen3-0.6B: lebih baru, multibahasa kuat
MODEL_NAME = "lora_qwen05b"      # GANTI tiap variasi (nama file OOF)
PROMPT = "{text}"                # teks apa adanya
# PROMPT = "Teks: {text}\nKategori:"     # awalan tugas: kadang +0.5 poin pada LLM instruct
MAX_LENGTH = 256                 # LLM boleh lebih panjang dari BERT, tapi VRAM naik
BATCH_SIZE = 8
ACCUM_STEPS = 2                  # batch efektif 16
EPOCHS = 2                       # LoRA cepat konvergen: 1-3
LEARNING_RATE = 2e-4             # LoRA butuh lr jauh lebih besar dari fine-tune penuh (1e-4..3e-4)
WARMUP = 0.1
LORA_R = 16                      # rank: 8 (hemat) .. 64 (data besar)
LORA_ALPHA = 32                  # skala = alpha / r
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
# LORA_TARGETS = ["q_proj", "v_proj"]   # LoRA klasik: lebih hemat, sedikit lebih lemah
GRAD_CHECKPOINT = True           # hemat VRAM, ± 30% lebih lambat
FOLDS_TO_RUN = None              # None = semua fold. Waktu mepet: [0] (OOF tidak disimpan)
SEED = 42

## Metrik

In [ ]:
METRIC_NAME = "f1_macro"         # GANTI saat lomba, ikut baris return di bawah


def score(truth, predicted):
    return f1_score(truth, predicted, average="macro")


print("metrik:", METRIC_NAME, "| device:", DEVICE)

## Baca data
Teks kosong/NaN jadi string kosong, lalu `clean_basic` (html + spasi rapi). Label diubah ke indeks
0..K-1 urut `np.unique` (format OOF bersama).

In [ ]:
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(OOF_FOLDER, exist_ok=True)
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)


def clean_basic(text):
    return " ".join(html.unescape(text).split())


train_text = train[TEXT_COL].fillna("").astype(str)
test_text = test[TEXT_COL].fillna("").astype(str)
train_text = train_text.apply(clean_basic)
test_text = test_text.apply(clean_basic)
classes = np.unique(train[TARGET])
y = np.searchsorted(classes, train[TARGET].to_numpy())
print("train:", train.shape, "| test:", test.shape, "| NaN target:", train[TARGET].isna().sum())
print("kelas:", list(classes), "| jumlah:", np.bincount(y).tolist())
train[[ID_COL, TEXT_COL, TARGET]].head()

## Buat sample_submission demo (HAPUS saat lomba)
Data latihan tidak punya `sample_submission`. Saat lomba file panitia dipakai (`SAMPLE_CSV`);
sel ini tidak menimpa file di luar `OUTPUT_FOLDER`.

In [ ]:
demo_sample = pd.DataFrame()
demo_sample[ID_COL] = test[ID_COL]
demo_sample[SUBMIT_COL] = train[TARGET].iloc[0]
if SAMPLE_CSV.startswith(OUTPUT_FOLDER):
    demo_sample.to_csv(SAMPLE_CSV, index=False)
    print("sample demo ditulis ke", SAMPLE_CSV)
else:
    print("SAMPLE_CSV bukan file demo: dibiarkan apa adanya")

In [ ]:
sample = pd.read_csv(SAMPLE_CSV)
print("sample_submission:", sample.shape, sample.columns.tolist())
print("id test = id sample:", set(test[ID_COL]) == set(sample[ID_COL]))
sample.head(3)

## Split (k-fold)
Aturan format OOF: `StratifiedKFold` acak seed `SPLIT_SEED` pada urutan baris train apa adanya,
jadi OOF notebook ini sefold dengan `finetune_oof`, `baseline_oof`, dll. dan bisa di-ensemble.

In [ ]:
splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SPLIT_SEED)
folds = list(splitter.split(train_text, y))
val_sizes = []
for _, val_index in folds:
    val_sizes.append(len(val_index))
print("fold:", len(folds), "| ukuran validasi:", val_sizes)

## Tokenisasi
LLM tidak punya token [CLS]; model membaca token terakhir yang bukan padding. Token padding
disamakan dengan token akhir teks bila tokenizer tidak punya.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(PRETRAINED, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


def encode_rows(texts):
    prompted = []
    for text in texts:
        prompted.append(PROMPT.format(text=text))
    encoded = tokenizer(prompted, truncation=True, max_length=MAX_LENGTH)
    rows = []
    for number in range(len(prompted)):
        rows.append({"input_ids": encoded["input_ids"][number],
                     "attention_mask": encoded["attention_mask"][number]})
    return rows


train_rows = encode_rows(train_text)
test_rows = encode_rows(test_text)
lengths = []
for row in train_rows:
    lengths.append(len(row["input_ids"]))
print("token p50/p95/maks:", np.percentile(lengths, [50, 95]).round(), max(lengths))

## LoRA buatan sendiri
`LoraLinear` = Linear asli (beku) + `up(down(x)) * alpha / r`. `up` mulai dari nol, jadi model
awal persis sama dengan pretrained. Hanya `down`, `up`, dan kepala yang dilatih (fp32).

In [ ]:
class LoraLinear(nn.Module):
    def __init__(self, base, rank, alpha, dropout):
        super().__init__()
        self.base = base
        self.down = nn.Linear(base.in_features, rank, bias=False)
        self.up = nn.Linear(rank, base.out_features, bias=False)
        nn.init.kaiming_uniform_(self.down.weight, a=5 ** 0.5)
        nn.init.zeros_(self.up.weight)
        self.drop = nn.Dropout(dropout)
        self.scale = alpha / rank

    def forward(self, inputs):
        return self.base(inputs) + self.up(self.down(self.drop(inputs))) * self.scale


print("LoraLinear siap")

In [ ]:
def add_lora(model):
    for param in model.parameters():
        param.requires_grad = False
    names = []
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and name.split(".")[-1] in LORA_TARGETS:
            names.append(name)
    for name in names:
        parent_name, child_name = name.rsplit(".", 1)
        parent = model.get_submodule(parent_name)
        wrapped = LoraLinear(getattr(parent, child_name), LORA_R, LORA_ALPHA, LORA_DROPOUT)
        setattr(parent, child_name, wrapped)
    head = model.score                 # kepala klasifikasi model decoder bernama `score`
    head.float()
    for param in head.parameters():
        param.requires_grad = True
    return len(names)


print("add_lora siap | target:", LORA_TARGETS)

In [ ]:
def make_model(seed):
    torch.manual_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        PRETRAINED, num_labels=len(classes), local_files_only=True, dtype=torch.bfloat16)
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    wrapped = add_lora(model)
    if GRAD_CHECKPOINT:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.enable_input_require_grads()
    model = model.to(DEVICE)
    trainable = 0
    for param in model.parameters():
        if param.requires_grad:
            trainable += param.numel()
    print(f"  Linear dibungkus LoRA: {wrapped} | parameter dilatih: {trainable / 1e6:.2f} juta")
    return model


print("make_model siap:", PRETRAINED)

## Latih dan prediksi
Autocast bf16 (juga di CPU). `lora_state` hanya menyimpan parameter yang dilatih (kecil), jadi
menyimpan bobot terbaik per epoch murah.

In [ ]:
def pick(rows, chosen):
    picked = []
    for number in chosen:
        picked.append(rows[number])
    return picked


def forward_logits(model, rows):
    batch = tokenizer.pad(rows, return_tensors="pt").to(DEVICE)
    with torch.autocast(DEVICE, dtype=torch.bfloat16):
        return model(**batch).logits.float()


def predict_proba(model, rows):
    model.eval()
    sizes = []
    for row in rows:
        sizes.append(len(row["input_ids"]))
    order = np.argsort(sizes)
    proba = np.zeros((len(rows), len(classes)), dtype="float32")
    with torch.no_grad():
        for start in range(0, len(order), BATCH_SIZE * 2):
            chosen = order[start:start + BATCH_SIZE * 2]
            proba[chosen] = softmax(forward_logits(model, pick(rows, chosen)), dim=1).cpu().numpy()
    return proba


print("forward / predict siap")

In [ ]:
def lora_state(model):
    state = {}
    for name, param in model.named_parameters():
        if param.requires_grad:
            kept = param.detach().cpu()
            state[name] = kept.clone()
    return state


def build_run(fit_size, seed):
    model = make_model(seed)
    params = []
    for param in model.parameters():
        if param.requires_grad:
            params.append(param)
    optimizer = torch.optim.AdamW(params, lr=LEARNING_RATE, weight_decay=0.01)
    steps = max(1, int(np.ceil(fit_size / (BATCH_SIZE * ACCUM_STEPS)))) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP * steps), steps)
    return {"model": model, "params": params, "optimizer": optimizer, "scheduler": scheduler,
            "best": -1.0, "state": None, "rng": np.random.default_rng(seed)}


print("build_run siap")

In [ ]:
def train_epoch(run, fit_index):
    model = run["model"]
    model.train()
    order = run["rng"].permutation(fit_index)
    for number, start in enumerate(range(0, len(order), BATCH_SIZE)):
        chosen = order[start:start + BATCH_SIZE]
        targets = torch.tensor(y[chosen]).to(DEVICE)
        loss = cross_entropy(forward_logits(model, pick(train_rows, chosen)), targets)
        (loss / ACCUM_STEPS).backward()
        if (number + 1) % ACCUM_STEPS == 0:
            clip_grad_norm_(run["params"], 1.0)
            run["optimizer"].step()
            run["scheduler"].step()
            run["optimizer"].zero_grad()


print("train_epoch siap")

In [ ]:
def train_fold(fit_index, val_index):
    run = build_run(len(fit_index), SEED)
    val_rows = pick(train_rows, val_index)
    for epoch in range(EPOCHS):
        began = time.time()
        train_epoch(run, fit_index)
        value = score(y[val_index], predict_proba(run["model"], val_rows).argmax(axis=1))
        if value > run["best"]:
            run["best"], run["state"] = value, lora_state(run["model"])
        print(f"  epoch {epoch + 1}/{EPOCHS}: {METRIC_NAME} {value:.4f} | "
              f"{time.time() - began:.0f} dtk")
    run["model"].load_state_dict(run["state"], strict=False)
    val_proba = predict_proba(run["model"], val_rows)
    test_part = predict_proba(run["model"], test_rows)
    del run
    torch.cuda.empty_cache()
    return val_proba, test_part


print("train_fold siap")

## Latih k-fold
Model dimuat ulang tiap fold (LoRA mulai dari nol). Test = rata-rata model tiap fold.

In [ ]:
folds_to_run = FOLDS_TO_RUN
if folds_to_run is None:
    folds_to_run = list(range(len(folds)))
oof = np.zeros((len(train), len(classes)), dtype="float32")
test_proba = np.zeros((len(test), len(classes)), dtype="float32")
if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()
began = time.time()
for position, number in enumerate(folds_to_run):
    fit_index, val_index = folds[number]
    val_proba, test_part = train_fold(fit_index, val_index)
    oof[val_index] = val_proba
    test_proba += test_part / len(folds_to_run)
    left = (time.time() - began) / (position + 1) * (len(folds_to_run) - position - 1)
    fold_value = score(y[val_index], val_proba.argmax(axis=1))
    print(f"fold {number}: {METRIC_NAME} {fold_value:.4f} | sisa ± {left / 60:.1f} menit")
peak_gb = 0.0
if DEVICE == "cuda":
    peak_gb = torch.cuda.max_memory_allocated() / 2**30
total_seconds = round(time.time() - began, 1)
note = (f"{os.path.basename(PRETRAINED)} LoRA r{LORA_R} | epoch {EPOCHS} lr {LEARNING_RATE} "
        f"len {MAX_LENGTH} | fold {folds_to_run} | VRAM {peak_gb:.2f} GB")
print(note, f"| total {total_seconds / 60:.1f} menit")

## Evaluasi OOF
Skor dihitung di baris train yang punya prediksi OOF (semua baris bila semua fold jalan).

In [ ]:
filled = oof.sum(axis=1) > 0
oof_pred = oof.argmax(axis=1)
oof_score = score(y[filled], oof_pred[filled])
print(f"{METRIC_NAME} OOF ({filled.sum()} baris): {oof_score:.4f}")
print(classification_report(y[filled], oof_pred[filled], target_names=classes.astype(str)))

## Simpan OOF dan catat percobaan
`oof_<MODEL_NAME>.npy` / `test_<MODEL_NAME>.npy` hanya bila semua baris train terisi, untuk
`ensemble_oof`. `percobaan.csv` bertambah satu baris tiap run.

In [ ]:
if filled.all():
    np.save(f"{OOF_FOLDER}/oof_{MODEL_NAME}.npy", oof)
    np.save(f"{OOF_FOLDER}/test_{MODEL_NAME}.npy", test_proba)
    print("OOF dan test disimpan:", MODEL_NAME)
else:
    print("OOF tidak lengkap (FOLDS_TO_RUN?): tidak disimpan")
row = {"waktu": time.strftime("%Y-%m-%d %H:%M:%S"), "notebook": NOTEBOOK, "model": MODEL_NAME,
       "metrik": METRIC_NAME, "skor_cv": round(oof_score, 4), "detik": total_seconds,
       "catatan": note}
log_path = f"{OOF_FOLDER}/percobaan.csv"
pd.DataFrame([row]).to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
history = pd.read_csv(log_path)
print(history.tail(5).to_string())

## Submission
Kolom id diambil dari `sample_submission`; prediksi dipetakan lewat id.

In [ ]:
test_pred = test_proba.argmax(axis=1)
pred_by_id = pd.Series(classes[test_pred], index=test[ID_COL])

submission = pd.DataFrame()
submission[ID_COL] = sample[ID_COL]
submission[SUBMIT_COL] = sample[ID_COL].map(pred_by_id)
submission.to_csv(f"{OUTPUT_FOLDER}/{SUBMISSION_NAME}", index=False)
predicted_share = submission[SUBMIT_COL].value_counts(normalize=True)
print("sebaran prediksi test:", predicted_share.round(3).to_dict())
submission.head()

## Cek submission terhadap sample_submission
Salinan validator `arsenal/umum/simpan_muat` #15 dan #16: hanya membaca dan mencetak. Hasil
akhir harus `MASALAH: 0`.

In [ ]:
SUBMISSION_CSV = f"{OUTPUT_FOLDER}/{SUBMISSION_NAME}"
ID_COLUMN = ID_COL

sample = pd.read_csv(SAMPLE_CSV)
written = pd.read_csv(SUBMISSION_CSV)
problems = []


def report(failed, text):
    if failed:
        problems.append(text)


wanted = sample[ID_COLUMN]
ids = written.get(ID_COLUMN, pd.Series(dtype=wanted.dtype))
lost = set(wanted) - set(ids)
extra = set(ids) - set(wanted)
kept = ids[ids.isin(wanted)].drop_duplicates().tolist()
expected = wanted[wanted.isin(ids)].tolist()
report(written.columns.tolist() != sample.columns.tolist(), "kolom / urutan kolom beda")
report(len(written) != len(sample), f"jumlah baris {len(written)}, sample {len(sample)}")
report(lost, f"id hilang {len(lost)}, mis. {sorted(lost)[:3]}")
report(extra, f"id asing {len(extra)}, mis. {sorted(extra)[:3]}")
report(ids.duplicated().any(), f"id kembar: {ids.duplicated().sum()} baris")
report(kept != expected, "urutan id beda dari sample")

In [ ]:
LABEL_COLUMNS = [SUBMIT_COL]     # kolom isi kelas ([] = lewati cek label)
ALLOWED = list(classes)          # label sah = label train, [] = lewati cek

for column in sample.columns:
    if column in written.columns:
        empty = written[column].isna().sum()
        report(empty, f"{column}: {empty} nilai kosong")
        report(written[column].dtype != sample[column].dtype,
               f"{column}: tipe {written[column].dtype}, sample {sample[column].dtype}")
for column in LABEL_COLUMNS:
    if ALLOWED and column in written.columns:
        foreign = set(written[column].dropna()) - set(ALLOWED)
        report(foreign, f"{column}: label di luar ALLOWED {sorted(foreign)[:5]}")
        print(written[column].value_counts())   # semua satu kelas -> curigai model
for problem in problems:
    print("-", problem)
print("MASALAH:", len(problems))

## Cek dengan kunci jawaban (HAPUS saat lomba)
Hanya untuk data latihan yang punya kunci (`ANSWER_CSV`). Hapus sel ini saat lomba.

In [ ]:
answers = pd.read_csv(ANSWER_CSV)
truth_by_id = pd.Series(answers[TARGET].to_numpy(), index=answers[ID_COL])
truth_index = np.searchsorted(classes, truth_by_id.reindex(submission[ID_COL]).to_numpy())
pred_index = np.searchsorted(classes, submission[SUBMIT_COL].to_numpy())
print(f"{METRIC_NAME} di test (kunci): {score(truth_index, pred_index):.4f}")
print(f"{METRIC_NAME} OOF train: {oof_score:.4f}")